In [1]:
from pprint import pprint

from dotenv import load_dotenv

load_dotenv()

True

## Requirements

1. Travel Agent: Finds flights to and from the destination
2. Venue Agent: Searches the web for a wedding venue
3. DJ Agent: Finds the playlist to match the right genre
4. Main Co-ordinator: Works with other agents to facilitate all this.

## Setup Tools

In [2]:
from langchain_mcp_adapters.client import  MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server":{
            "transport": "streamable_http",
            "url": "https://mcp.kiwi.com"
        }
    }
)

tools = await client.get_tools()

In [3]:
tools

[StructuredTool(name='search-flight', description='\n# Search for a flight\n\n## Description\n\nUses the Kiwi API to search for available flights between two locations on a specific date.\n\n## How it works\n\nThe tool will:\n1. Search for matching locations to resolve airport codes\n2. Find available flights for the specified route and date range\n\n## Method\n\nCall this tool whenever a user wants to search for flights, regardless of whether they provided exact airport codes or just city names.\n\nYou should display the returned results in a markdown table format: Group the results by price (those who are the cheapest), duration (those who are the shortest, i.e. have the smallest \'totalDurationInSeconds\') and the rest (those that could still be interesting).\n\nAlways display for each flight in order:\n  - In the 1st column: The departure and arrival airports, including layovers (e.g. "Paris CDG → Barcelona BCN → Lisbon LIS")\n  - In the 2nd column: The departure and arrival dates 

In [4]:
from typing import Dict, Any
from tavily import TavilyClient
from langchain.tools import tool

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""

    return tavily_client.search(query)

In [5]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def query_playlist_db(query: str) -> str:
    """Query the database for playlist information"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"

## Create State

In [6]:
from langchain.agents import AgentState

class WeddingState(AgentState):
    origin: str
    destination: str
    guest_count: str
    genre: str


## Create Subagents

In [7]:
from langchain.agents import create_agent

# Travel Agent
travel_agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt="""
    You are a travel agent. Search for flights to the desired destination wedding location. You are not allowed to ask any more follow up questions, you must find the best flight options based on the following criteria:
        - Price (lowest, economy class)
        - Duration (shortest)
        - Date (time of year which you believe is best for a wedding at this location)
    To make things easy, only look for one ticket, one way.
    You may need to make multiple searches to iteratively find the best options.
    You will be given no extra information, only the origin and destination. It is your job to think critically about the best options.
    Once you have found the best options, let the user know your shortlist of options.
    """
)

In [8]:
# Venue Agent

venue_agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    system_prompt="""
    You are a venue specialist. Search for venues in the desired location, and with the desired capacity. You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
        - Price (lowest)
        - Capacity (exact match)
        - Reviews (highest)

    You may need to make multiple searches to iteratively find the best options.
    """
)

In [9]:
# Playlist agent
playlist_agent = create_agent(
    model="gpt-5-nano",
    tools=[query_playlist_db],
    system_prompt="""
    You are a playlist specialist. Query the SQL database and curate the perfect playlist for a wedding given a genre. Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price. If you run into errors when querying the database, try to fix them by making changes to the query. Do not come back empty handed, keep trying to query the DB until you find a list of songs. You may need to make multiple searches to iteratively find the best options.
    """
)

## Main Coordinator

In [10]:
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command


@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """Travel agent searches for flights to the desired destination wedding location"""
    origin = runtime.state["origin"]
    destination = runtime.state["destination"]
    response = await travel_agent.ainvoke({"messages": [HumanMessage(content=f"Find flights from {origin} to {destination}")]})
    return response["messages"][-1].content

@tool
def search_venues(runtime: ToolRuntime) -> str:
    """Venue agent chooses the best venue for the given location and capacity"""
    destination = runtime.state["destination"]
    capacity = runtime.state["guest_count"]
    query = f"Find wedding venues in {destination} for {capacity} guests"
    response = venue_agent.invoke({"messages":[HumanMessage(content=query)]})
    return response["messages"][-1].content


@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """Playlist agent curates the perfect playlist for the given genre"""
    genere = runtime.state["genre"]
    query = f"Find {genere} tracks for the wedding playlist"
    response = playlist_agent.invoke({"messages":[HumanMessage(content=query)]})
    return response["messages"][-1].content

@tool
def update_state(origin: str,destination: str, guest_count: str, genre: str, runtime: ToolRuntime) -> str:
    """Update the state when you know all of the values: origin, destination, guest_count, genre"""
    return Command(update={
        "origin": origin,
        "destination": destination,
        "guest_count": guest_count,
        "genre": genre,
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]}
    )

In [11]:
from langchain.agents import create_agent

coordinator = create_agent(
    model="gpt-5-nano",
    tools=[search_flights, search_venues, suggest_playlist, update_state],
    state_schema=WeddingState,
    system_prompt="""
    You are a wedding coordinator. Delegate tasks to your specialists for flights, venues and playlists.
    First find all the information you need to update the state. Once that is done you can delegate the tasks.
    Once you have received their answers, coordinate the perfect wedding for me.
    """
)

## Test

In [12]:
from langchain.messages import HumanMessage

response = await coordinator.ainvoke(
    {"messages": [HumanMessage(content="I am from Bengaluru and I would like a wedding in Maldives for 100 guests, with music to be of jazz genre")]}
)

In [14]:
from pprint import pprint

In [15]:
pprint(response)

{'destination': 'Maldives',
 'genre': 'Jazz',
 'guest_count': '100',
 'messages': [HumanMessage(content='I am from Bengaluru and I would like a wedding in Maldives for 100 guests, with music to be of jazz genre', additional_kwargs={}, response_metadata={}, id='3ada062f-d54a-4116-94df-35539a78ea5d'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 296, 'prompt_tokens': 299, 'total_tokens': 595, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-D7kflLLYSKgcsMiFGqiYVhxwQEjEG', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c4855-5707-7181-82b6-3cce9177aed9-0', tool_calls=[{'name': 'update_state', 'arg

In [16]:
pprint(response["messages"][-1].content)

('Fantastic news. I’ve pulled together tailored options for flights, venues, '
 'and a Jazz-inspired playlist for your Maldives wedding with 100 guests. '
 'Here’s a clear plan to move forward.\n'
 '\n'
 '1) Flights from Bengaluru (BLR) to Malé (MLE)\n'
 'Options (economy, one-way) to fit a wedding timeline:\n'
 '- Option A: BLR → MLE via Kochi (COK)\n'
 '  - Dep: 01 Mar 2026 05:20 local → Arr: 01 Mar 2026 18:55 local\n'
 '  - Duration: ~14h 05m\n'
 '  - Price: EUR 123\n'
 '  - Booking: https://on.kiwi.com/hPr5pk\n'
 '  - Pros: Lowest price, straightforward connections\n'
 '- Option B: BLR → MLE via Trivandrum (TRV)\n'
 '  - Dep: 01 Mar 2026 08:55 → Arr: 01 Mar 2026 18:55\n'
 '  - Duration: ~10h 30m\n'
 '  - Price: EUR 123\n'
 '  - Booking: https://on.kiwi.com/SPB2mE\n'
 '  - Pros: Shorter travel time\n'
 '- Option C: BLR → MLE via Kochi (COK)\n'
 '  - Dep: 15 Mar 2026 13:05 → Arr: 15 Mar 2026 18:55\n'
 '  - Duration: ~10h 30m\n'
 '  - Price: EUR 123\n'
 '  - Booking: https://on.kiwi.c

In [19]:
 from IPython.display import Markdown, display

In [20]:
display(Markdown(response["messages"][-1].content))

Fantastic news. I’ve pulled together tailored options for flights, venues, and a Jazz-inspired playlist for your Maldives wedding with 100 guests. Here’s a clear plan to move forward.

1) Flights from Bengaluru (BLR) to Malé (MLE)
Options (economy, one-way) to fit a wedding timeline:
- Option A: BLR → MLE via Kochi (COK)
  - Dep: 01 Mar 2026 05:20 local → Arr: 01 Mar 2026 18:55 local
  - Duration: ~14h 05m
  - Price: EUR 123
  - Booking: https://on.kiwi.com/hPr5pk
  - Pros: Lowest price, straightforward connections
- Option B: BLR → MLE via Trivandrum (TRV)
  - Dep: 01 Mar 2026 08:55 → Arr: 01 Mar 2026 18:55
  - Duration: ~10h 30m
  - Price: EUR 123
  - Booking: https://on.kiwi.com/SPB2mE
  - Pros: Shorter travel time
- Option C: BLR → MLE via Kochi (COK)
  - Dep: 15 Mar 2026 13:05 → Arr: 15 Mar 2026 18:55
  - Duration: ~10h 30m
  - Price: EUR 123
  - Booking: https://on.kiwi.com/NfZ6Ld
  - Pros: Flexible mid-March date, reasonable travel time

Notes:
- Maldives weather is typically best in late February to early March; if you’re flexible, we can target that window for nicer conditions and good prices.
- I can search additional date windows or different carriers to optimize for weather and cost. Do you have exact wedding dates in mind?

2) Venues in Maldives for 100 guests
Here are solid options with capacity fit, rough package context, and vibe notes:

- 1) Sheraton Maldives Full Moon Resort & Spa (Furanafushi Island, North Malé Atoll)
  - Capacity: Jalsaa Room fits up to 100 guests
  - Price indicators: Basic wedding package around USD 850 (ceremony-focused); full packages vary and are typically higher
  - Vibe: Classic, reliable luxury with strong wedding infrastructure
  - Sources for details: capacity and package notes (venues): 
    - https://www.marriott.com/en-us/hotels/mlesi-sheraton-maldives-full-moon-resort-and-spa/events/
    - https://www.asiadreams.com/sheraton-maldives-full-moon-resort-spa/

- 2) Anantara Dhigu Maldives Resort (Gulhifushi Private Island)
  - Capacity: Reported to host up to 100 guests
  - Price indicators: Example: USD 23,800 for 30 guests (scales with guest count; exact 100-guest package not publicly listed)
  - Vibe: Modern luxury, private-island feel, excellent service
  - Sources: 
    - https://www.anantara.com/en/dhigu-maldives/weddings
    - https://www.myoverseaswedding.com/wedding-destinations/maldives/anantara-dhigu-resort-spa/reception-venues

- 3) Anantara Kihavah Maldives Villas (Baa Atoll)
  - Capacity: Around 100 guests
  - Price indicator: About USD 67,300 for a 100-guest package (example package)
  - Vibe: Ultra-luxury, striking overwater charters, stunning culinary options
  - Sources:
    - https://www.myoverseaswedding.com/wedding-packages/maldives/100-guests/

- 4) Soneva Fushi (Baa Atoll)
  - Capacity: Listed as able to host ~100 guests
  - Price indicator: High-end; no flat-rate public package; pricing premium per guest
  - Vibe: Ultra-luxury, eco-luxe, barefoot-chic feel
  - Source: Venue reports
    - https://www.venuereport.com/find-venues/maldives/baa-atoll/wedding/

Notes on venue planning:
- Most Maldivian resorts offer symbolic ceremonies by default; legal marriage generally requires steps outside the resort. I can summarize the usual process for the exact destinations you’re considering if you’d like.
- Packages for 100 guests vary widely and often require a customizable quote. I can push for formal proposals with inclusions (ceremony setup, reception dinner, decor, music, photographer, etc.) once you pick 2–3 favorites.

3) Jazz wedding playlist (curated for reception)
Selected jazz tracks tailored for romance, swing, and easy dancing. Duration and cost provided:
- Total length: 57 minutes 9 seconds
- Total cost: USD 11.88
1) Round Midnight — Miles Davis
2) Angela — Antonio Carlos Jobim
3) Canta, Canta Mais — Antonio Carlos Jobim
4) Corcovado (Quiet Nights Of Quiet Stars) — Antonio Carlos Jobim
5) Bye Bye Blackbird — Miles Davis
6) Blues For Pablo — Miles Davis
7) Best Thing — Spyro Gyra
8) Believe — Spyro Gyra
9) As We Sleep — Spyro Gyra
10) Blue Rhythm Fantasy — Gene Krupa
11) Boogie Blues — Gene Krupa
12) Coronation Drop — Gene Krupa

Notes:
- I can swap in more upbeat dance tracks, add more bossa nova, or extend the set for longer reception needs.
- Tell me your preferred mood (romantic, lounge, or upbeat dance) and I’ll tailor the set length and tempo.

Next steps I can take for you
- Flight hold or booking: I can pin down your preferred flight option (A, B, or C) and place a hold for a short window while we finalize dates, passenger details, and payment.
- Venue quotes: I can reach out to the top 2–3 venues (Sheraton Full Moon, Anantara Dhigu, Anantara Kihavah, Soneva Fushi) to request formal packages for 100 guests on your target dates, including ceremony, reception, decor, catering, and sound/music options.
- Playlist customization: I can adjust the length and tempo to fit a 4–5 hour reception or extend to a full-night dance set.

Questions to finalize
- Do you have fixed wedding dates, or should I search a date window around late February to early March 2026 for optimal weather and price?
- Which flight option (A, B, or C) would you prefer to hold, and what rough dates would you like to target?
- Which 2–3 venues feel like the best fit for your vibe (classic luxury, private island feel, ultra-luxury)? I’ll request formal proposals from those.
- Any dietary restrictions, guest room block needs, or accessibility considerations we should plan for?
- Do you want a symbolic ceremony only, or should I include a plan for the legal/process steps if you want that handled through the resort or a local wedding planner?

If you’d like, I can proceed now with:
- Holding a flight option (e.g., Option B for ~10h30m) while we confirm dates
- Sending formal venue quotes for your top 2–3 choices
- Customizing the Jazz playlist to extend into a longer reception

Tell me your preferred dates and which venues you want me to contact first, and I’ll take it from there.